# FabricSimCity — capacity metrics ingest

Copies the Capacity Metrics semantic model into the FabricSimCity app's SQL database on a schedule,
because a Fabric App has no timers of its own and the semantic model can only be read with the
signed-in user's own permissions.

**Before the first run**

1. Deploy the app with `npx rayfin up` so the `IngestRuns` and `IngestRows` tables exist.
2. Give this notebook's identity access to the app's SQL database, as a user with
   `INSERT`, `UPDATE`, `DELETE` and `SELECT` on those two tables.
3. Give it Read and Build access to the Capacity Metrics semantic model, plus permission to
   discover model metadata for the schema probe. Report viewing alone is insufficient.
4. Fill in the parameters cell below.

Then schedule it at the same interval you set `VITE_FABRIC_INGEST_INTERVAL_MINUTES` to.
Scheduled runs use the identity of the user who created or last updated the schedule.

**Updating an existing notebook:** this notebook requires manifest version **3**. Update both
the notebook and its Built-in `dax-queries.generated.json`, preserving your parameter values,
then restart the session and run all cells. The SQL entity schema and stored summary layout have
not changed; existing ingested readers remain compatible. Confirm the schedule targets this
updated notebook. App deployments use `VITE_FABRIC_SOURCE=ingested` and the matching tenant
and dataset ids. The app reads through Rayfin's built-in GraphQL adapter and adopts the Fabric portal
session automatically, without a separate login flow.

**Daily metrics with dimensions:** `metricsDailyWithDimensions` reads the daily fact plus
`Items` and `Capacities`. Imported capacity metadata supplies routing first, then each fact query
binds `MPARAMETER 'CapacitiesList'` and `MPARAMETER 'RegionName'`. A DAX row filter alone does
not populate this DirectQuery source. Routing uses `Region without default`, not the display
label `Default`. It supplies CU, durations, operation counts and measured throttling,
not 30-second utilization or per-item OneLake storage. Those unavailable measurements remain
unknown. Autoscale-specific facts are not combined. The first partial day is excluded; freshness
uses the latest daily bucket, not the time this notebook ran.

**If ADOMD reports no permission to call Discover:** check the executing identity's access to
the model in `METRICS_WORKSPACE_ID`, not the notebook or app workspace. After running the
parameters cell, try `evaluate_dax('EVALUATE ROW("AccessCheck", 1)')` separately from the
ingest. If it also fails, check Read/Build permissions and XMLA access. If it succeeds but the
schema probe fails, ordinary querying works but metadata discovery does not.
[Microsoft documents model-admin permissions for INFO metadata queries](https://learn.microsoft.com/dax/info-functions-dax).
Use an authorized model administrator for that operation; do not grant tenant-wide admin or
change SQL permissions to work around it.

**If no known schema generation matches:** the error distinguishes an absent table from missing
required columns. The notebook saves the complete table/column map to
`builtin/capacity-metrics-schema.json`; download it from Resources > Built-in to diagnose the
adapter mismatch. It contains names only, not metric values or credentials. Ingest stops before
any SQL write. Do not rename model tables or skip validation to force a match.

**Everyone signed in to the app can read everything this writes.** The semantic model checks each
user's own capacity permissions; a table does not. Ingest only capacities your app's users are
all entitled to see.

## Pure logic

Generated from `fabric/simcity_ingest.py`. Edit that file, not this cell, and run
`npm run fabric:notebook` — `ingestNotebook.test.ts` fails if they disagree.

In [ ]:
"""
Pure logic for the FabricSimCity capacity metrics ingest.

Kept free of every Fabric import so it can be executed and tested on an ordinary machine — see
`src/collect/ingestNotebook.test.ts`, which runs this file under Python and checks the two
substitution traps below. The notebook cell that does I/O imports from here rather than restating
any of it.

Two things in here fail as wrong numbers rather than as an error, which is why they are tested:

* `bind_dax_parameters` matches whole identifiers. A plain string replace of `@Start` would also
  rewrite the front of `@StartOfDay`, and what it leaves behind is still valid DAX.
* `dax_literal` doubles quotes in strings, which is DAX's own escape, so a capacity id cannot end
  the literal and carry on as expression text.

Both mirror `src/collect/semanticModelDaxClient.ts`. Change one, change the other.
"""

from __future__ import annotations

import datetime as _dt
import json
import re
from typing import Any, Iterable, Mapping, Sequence

# Matches `@Name` as a whole identifier. See the module docstring for why this is not a plain
# replace of each parameter name in turn.
_PARAMETER_REFERENCE = re.compile(r"@([A-Za-z_][A-Za-z0-9_]*)")

# Accepts the ISO-8601 shapes the app emits. Anything else is treated as an ordinary string, so a
# capacity name that merely looks date-ish is still quoted rather than turned into arithmetic.
_ISO_TIMESTAMP = re.compile(
    r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}(:\d{2})?(\.\d+)?(Z|[+-]\d{2}:\d{2})$"
)

# `executeQueries` and `sempy` both return bracketed column names: `[CapacityId]` for a
# SELECTCOLUMNS alias, `Table[Column]` for a bare column reference.
_BRACKETED_KEY = re.compile(r"\[([^\[\]]+)\]$")


class IngestError(RuntimeError):
    """Raised for a condition the operator has to fix, rather than one to retry."""


def parse_iso(value: str) -> _dt.datetime | None:
    """Parse an ISO timestamp to an aware UTC datetime, or return None if it is not one."""
    if not isinstance(value, str) or not _ISO_TIMESTAMP.match(value):
        return None
    try:
        parsed = _dt.datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return None
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=_dt.timezone.utc)
    return parsed.astimezone(_dt.timezone.utc)


def dax_literal(value: Any) -> str:
    """Render a Python value as a DAX literal.

    Timestamps become `DATE(...) + TIME(...)` rather than a quoted string because DAX comparison
    against text is a type error in some models and silently coerces in others; built
    arithmetically it is unambiguous in both.
    """
    if value is None:
        return "BLANK()"
    if isinstance(value, bool):
        return "TRUE()" if value else "FALSE()"
    if isinstance(value, (int, float)):
        if isinstance(value, float) and (value != value or value in (float("inf"), float("-inf"))):
            raise IngestError(f"Cannot bind non-finite number {value!r} into DAX.")
        return repr(value) if isinstance(value, float) else str(value)
    if isinstance(value, _dt.datetime):
        return _timestamp_literal(value)

    text = str(value)
    timestamp = parse_iso(text)
    if timestamp is not None:
        return _timestamp_literal(timestamp)
    escaped = text.replace('"', '""')
    return f'"{escaped}"'


def _timestamp_literal(value: _dt.datetime) -> str:
    utc = value.astimezone(_dt.timezone.utc) if value.tzinfo else value.replace(tzinfo=_dt.timezone.utc)
    return (
        f"(DATE({utc.year},{utc.month},{utc.day})"
        f" + TIME({utc.hour},{utc.minute},{utc.second}))"
    )


def bind_dax_parameters(query: str, parameters: Mapping[str, Any]) -> str:
    """Substitute `@Name` placeholders with DAX literals.

    An unbound placeholder is a bug in the generated query, so it raises rather than being left in
    the text for the model to reject with a less specific message.
    """

    def replace(match: re.Match[str]) -> str:
        name = match.group(1)
        if name not in parameters:
            raise IngestError(
                f"DAX query references {match.group(0)} but no such parameter was supplied."
            )
        return dax_literal(parameters[name])

    return _PARAMETER_REFERENCE.sub(replace, query)


def normalize_row_key(key: str) -> str:
    """Strip the brackets DAX transports wrap column names in."""
    match = _BRACKETED_KEY.search(key)
    return match.group(1) if match else key


def normalize_row(row: Mapping[str, Any]) -> dict[str, Any]:
    return {normalize_row_key(str(key)): value for key, value in row.items()}


def probe_tables(rows: Iterable[Mapping[str, Any]]) -> dict[str, set[str]]:
    """Turn schema-probe rows into a table name -> column names map.

    Mirrors `schemaTables` in `semanticModelSource.ts`, including its fallback from the
    `TableName`/`ColumnName` aliases to the bare `Table`/`Name` columns of `INFO.VIEW.COLUMNS()`.
    """
    tables: dict[str, set[str]] = {}
    for raw in rows:
        row = normalize_row(raw)
        table = row.get("TableName", row.get("Table"))
        column = row.get("ColumnName", row.get("Name"))
        if not isinstance(table, str) or not isinstance(column, str):
            continue
        tables.setdefault(table, set()).add(column)
    return tables


def pick_generation(
    manifest: Mapping[str, Any],
    probe_rows: Sequence[Mapping[str, Any]],
) -> Mapping[str, Any]:
    """Choose the manifest entry matching the model, by the same rule the app applies.

    Selecting differently to `createSemanticModelSource` would be the worst kind of mismatch: the
    notebook would ingest one generation's rows and the app would parse them as the other's, which
    reads as a model with every value missing rather than as an error.
    """
    tables = probe_tables(probe_rows)
    mismatches: list[str] = []
    for generation in manifest["generations"]:
        required_tables = generation.get(
            "requiredTables", {generation["table"]: generation["requiredColumns"]}
        )
        missing_tables: list[str] = []
        for table, required in required_tables.items():
            columns = tables.get(table)
            if columns is None:
                missing_tables.append(f"{table}: table absent")
            else:
                missing = sorted(set(required) - columns)
                if missing:
                    missing_tables.append(f"{table}: missing columns {', '.join(missing)}")
        if not missing_tables:
            return generation
        mismatches.append(f"{generation['name']} ({'; '.join(missing_tables)})")

    seen = ", ".join(sorted(tables)[:10]) or "no tables at all"
    preview = " (first 10)" if len(tables) > 10 else ""
    raise IngestError(
        "Capacity Metrics semantic model matched no known schema generation. "
        f"Expected schemas: {'; '.join(mismatches)}. "
        f"The probe returned {len(tables)} tables{preview}: {seen}."
    )


def probe_rows_for_table(rows: Iterable[Mapping[str, Any]], table: str) -> list[dict[str, Any]]:
    """Keep only the probe rows describing the table the app will look up.

    `INFO.VIEW.COLUMNS()` describes the entire model. The generation helper calls this for each
    required fact or dimension table so unrelated tables do not multiply the size of every ingest.
    """
    kept: list[dict[str, Any]] = []
    for raw in rows:
        row = normalize_row(raw)
        name = row.get("TableName", row.get("Table"))
        if name == table:
            kept.append(row)
    return kept


def probe_rows_for_generation(
    rows: Sequence[Mapping[str, Any]], generation: Mapping[str, Any]
) -> list[dict[str, Any]]:
    """Keep dimensions as well as the fact table so the reader can detect the same schema."""
    tables = generation.get("requiredTables", {generation["table"]: generation["requiredColumns"]})
    return [row for table in tables for row in probe_rows_for_table(rows, table)]


def capacity_ids(rows: Iterable[Mapping[str, Any]], capacity_id_column: str) -> list[str]:
    """Distinct capacity ids from the summary rows, in first-seen order."""
    seen: list[str] = []
    known: set[str] = set()
    for raw in rows:
        row = normalize_row(raw)
        value = row.get(capacity_id_column)
        if isinstance(value, str) and value and value not in known:
            known.add(value)
            seen.append(value)
    return seen


def _utc_datetime(value: _dt.datetime) -> _dt.datetime | None:
    # pandas.NaT is a datetime subclass that compares unequal to itself, not a usable timestamp.
    if value != value:
        return None
    aware = value if value.tzinfo else value.replace(tzinfo=_dt.timezone.utc)
    return aware.astimezone(_dt.timezone.utc)


def row_timestamp(row: Mapping[str, Any], timestamp_column: str) -> _dt.datetime | None:
    """Lift a row's own timepoint out, so the reader can window it in SQL."""
    value = normalize_row(row).get(timestamp_column)
    if isinstance(value, _dt.datetime):
        return _utc_datetime(value)
    if isinstance(value, str):
        return parse_iso(value)
    return None


def json_safe(value: Any) -> Any:
    """Coerce a DAX cell into something `json.dumps` accepts and the TypeScript parser reads back.

    Timestamps are rendered as ISO strings because that is what the app's contracts carry, and what
    `semanticModelSource` parses with `Date.parse`.
    """
    if value is None or isinstance(value, (str, bool, int)):
        return value
    if isinstance(value, float):
        return None if value != value else value
    if isinstance(value, _dt.datetime):
        timestamp = _utc_datetime(value)
        return timestamp.isoformat().replace("+00:00", "Z") if timestamp is not None else None
    if isinstance(value, _dt.date):
        return value.isoformat()
    return str(value)


# The `rowJson` column is NVARCHAR(3500). A row wider than that is a schema change worth failing
# on rather than truncating, because a truncated row is unparseable JSON that would surface much
# later as a corrupt-row error with no hint of where it came from.
ROW_JSON_LIMIT = 3500


def encode_row(row: Mapping[str, Any], query_name: str, row_index: int) -> str:
    payload = json.dumps({str(k): json_safe(v) for k, v in row.items()}, separators=(",", ":"))
    if len(payload) > ROW_JSON_LIMIT:
        raise IngestError(
            f"{query_name} row {row_index} serializes to {len(payload)} characters, over the "
            f"{ROW_JSON_LIMIT} the rowJson column holds."
        )
    return payload

## The DAX

Generated from the app's own query builder into `fabric/dax-queries.generated.json`, so the
notebook and the app can never ask the model different questions. Upload that file to this
notebook's built-in resources, or paste its contents into the cell below.

In [ ]:
import json
from pathlib import Path

# Fabric mounts a notebook's uploaded resources here.
MANIFEST_PATH = "/lakehouse/default/Files/dax-queries.generated.json"
_local = Path("./builtin/dax-queries.generated.json")

if _local.exists():
    MANIFEST = json.loads(_local.read_text(encoding="utf-8"))
elif Path(MANIFEST_PATH).exists():
    MANIFEST = json.loads(Path(MANIFEST_PATH).read_text(encoding="utf-8"))
else:
    raise IngestError(
        "Upload fabric/dax-queries.generated.json to this notebook's built-in resources, "
        f"or place it at {MANIFEST_PATH}."
    )

print(f"Manifest version {MANIFEST['version']} with "
      f"{len(MANIFEST['generations'])} schema generation(s).")

In [ ]:
"""
Fabric side of the FabricSimCity capacity metrics ingest.

Runs the DAX from `fabric/dax-queries.generated.json` against the Capacity Metrics semantic model
and writes the rows into the Rayfin app's SQL database, where the deployed app reads them back.

Everything the app needs to interpret those rows already exists in TypeScript, so this file does no
interpretation: it stores rows exactly as the model returned them. See `src/collect/ingestedDax.ts`.
"""

from __future__ import annotations

import datetime as _dt
import json
import struct
import uuid
from pathlib import Path

# In the notebook these come from the logic cell above, which has already run; outside it they come
# from the file that cell is generated from. Written this way so both the notebook and the tests
# execute exactly the same code.
if "bind_dax_parameters" not in globals():  # pragma: no cover - notebook path
    from simcity_ingest import (  # noqa: F401
        IngestError,
        bind_dax_parameters,
        capacity_ids,
        encode_row,
        normalize_row,
        pick_generation,
        probe_rows_for_generation,
        probe_tables,
        row_timestamp,
    )

# Must equal DAX_MANIFEST_VERSION in `src/collect/daxManifest.ts`. Pinned by `ingestNotebook.test.ts`.
DAX_MANIFEST_VERSION = 3

# PARAMETERS -------------------------------------------------------------------------------------
# Tag this cell "Parameters" in the Fabric notebook so a schedule or pipeline can override them.

# The Capacity Metrics semantic model. Find both ids in the Fabric portal URL when the model is
# open: /groups/<METRICS_WORKSPACE_ID>/datasets/<METRICS_DATASET_ID>. The metrics app installs into
# its own workspace, so this is not the workspace your app is deployed to.
METRICS_DATASET_ID = ""
METRICS_WORKSPACE_ID = ""

# The Rayfin app's SQL database. Copy from the SQL Database child item of your Fabric data app:
# Settings -> Connection strings. SQL_SERVER is the host only, with no `tcp:` prefix or port.
SQL_SERVER = ""
SQL_DATABASE = ""

# Written to every row. Must match what the app passes as its tenant, which is
# VITE_FABRIC_TENANT_ID, falling back to VITE_FABRIC_WORKSPACE_ID.
TENANT_ID = ""

# How much history to pull. Must match VITE_FABRIC_INGEST_WINDOW_DAYS in the app, which is what it
# declares as the source's retention. `ingestNotebook.test.ts` pins the two together.
INGEST_WINDOW_DAYS = 3

# How often this notebook is scheduled. Must match VITE_FABRIC_INGEST_INTERVAL_MINUTES, which the
# app adds to the model's own lag to report how stale the city may be.
INGEST_INTERVAL_MINUTES = 60

# Completed runs to keep. Older ones and their rows are deleted at the end of a successful run, so
# the table does not grow without bound. Keeping more than one means a run that fails midway leaves
# the previous city intact rather than emptying it.
KEEP_RUNS = 3

# Rows per INSERT batch.
BATCH_SIZE = 500
# ------------------------------------------------------------------------------------------------

SQL_RESOURCE = "https://database.windows.net/"
SQL_COPT_SS_ACCESS_TOKEN = 1256


def _now() -> _dt.datetime:
    return _dt.datetime.now(_dt.timezone.utc)


def _iso(value: _dt.datetime) -> str:
    return value.astimezone(_dt.timezone.utc).isoformat().replace("+00:00", "Z")


def connect_sql():
    """Open a token-authenticated connection to the Rayfin database.

    `notebookutils.credentials.getToken` has no documented audience key for SQL, but it accepts a
    resource URI, which is the pattern every Fabric-to-SQL sample uses. The token is passed through
    the ODBC connection attribute rather than the connection string because there is no
    connection-string form for a bearer token.
    """
    import pyodbc  # noqa: PLC0415 - notebook-only dependency
    from notebookutils import credentials  # noqa: PLC0415 - Fabric runtime only

    token = credentials.getToken(SQL_RESOURCE)
    encoded = b"".join(bytes([b]) + b"\x00" for b in token.encode("utf-8"))
    token_struct = struct.pack("=i", len(encoded)) + encoded

    driver = _newest_odbc_driver(pyodbc)
    connection_string = (
        f"Driver={{{driver}}};"
        f"Server={SQL_SERVER};"
        f"Database={SQL_DATABASE};"
        "Encrypt=yes;TrustServerCertificate=no;"
    )
    return pyodbc.connect(
        connection_string,
        attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct},
        autocommit=False,
    )


def _newest_odbc_driver(pyodbc) -> str:
    """Pick the newest installed msodbcsql.

    Pinning "ODBC Driver 18 for SQL Server" is the usual advice, but the Fabric runtime image is
    not something this repo controls, and a hard-coded name fails with a driver-not-found error
    that reads like a network problem.
    """
    drivers = [name for name in pyodbc.drivers() if "ODBC Driver" in name and "SQL Server" in name]
    if not drivers:
        raise RuntimeError(
            f"No msodbcsql driver is installed. pyodbc reports: {pyodbc.drivers()}"
        )
    return sorted(drivers)[-1]


def resolve_table(cursor, entity_name: str, required_columns: list[str]) -> str:
    """Find the table Rayfin generated for an entity, and check it has the columns expected.

    Rayfin pluralizes these entity names. Keep singular tables compatible, but never choose the
    first match when multiple schemas or naming generations coexist.
    """
    names = {
        "IngestRun": ("IngestRun", "IngestRuns"),
        "IngestRow": ("IngestRow", "IngestRows"),
    }.get(entity_name, (entity_name,))
    placeholders = ", ".join("LOWER(?)" for _ in names)
    cursor.execute(
        f"""
        SELECT TABLE_SCHEMA, TABLE_NAME
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_TYPE = 'BASE TABLE' AND LOWER(TABLE_NAME) IN ({placeholders})
        """,
        *names,
    )
    matches = cursor.fetchall()
    if not matches:
        cursor.execute(
            "SELECT TABLE_SCHEMA + '.' + TABLE_NAME FROM INFORMATION_SCHEMA.TABLES "
            "WHERE TABLE_TYPE = 'BASE TABLE' ORDER BY TABLE_NAME"
        )
        found = ", ".join(row[0] for row in cursor.fetchall()) or "no tables at all"
        raise RuntimeError(
            f"No table for entity {entity_name}. Run `npx rayfin up` to apply the schema. "
            f"The database currently has: {found}."
        )

    if len(matches) != 1:
        found = ", ".join(f"{schema}.{table}" for schema, table in matches)
        raise RuntimeError(
            f"Multiple tables match entity {entity_name}: {found}. "
            "Resolve the ambiguity before ingesting; no table was selected."
        )
    schema_name, table_name = matches[0]
    qualified = ".".join("[" + name.replace("]", "]]") + "]" for name in (schema_name, table_name))

    cursor.execute(
        "SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS "
        "WHERE TABLE_SCHEMA = ? AND TABLE_NAME = ?",
        schema_name,
        table_name,
    )
    present = {row[0].lower() for row in cursor.fetchall()}
    missing = [column for column in required_columns if column.lower() not in present]
    if missing:
        raise RuntimeError(
            f"{qualified} is missing {', '.join(missing)}. It was probably created by an older "
            "schema; run `npx rayfin up` to bring it up to date."
        )
    return qualified


def evaluate_dax(query: str):
    """Run one DAX query against the metrics model and return plain dict rows."""
    import sempy.fabric as fabric  # noqa: PLC0415 - Fabric runtime only

    frame = fabric.evaluate_dax(
        dataset=METRICS_DATASET_ID,
        dax_string=query,
        workspace=METRICS_WORKSPACE_ID or None,
    )
    return frame.to_dict(orient="records")


def check_parameters() -> None:
    missing = [
        name
        for name, value in (
            ("METRICS_DATASET_ID", METRICS_DATASET_ID),
            ("SQL_SERVER", SQL_SERVER),
            ("SQL_DATABASE", SQL_DATABASE),
            ("TENANT_ID", TENANT_ID),
        )
        if not value
    ]
    if missing:
        raise IngestError(f"Set these parameters before running: {', '.join(missing)}.")


def run_ingest(manifest: dict) -> str:
    """Ingest one snapshot and return the run id.

    The run row is written first as `Running` and flipped to `Complete` only once every row is in,
    because the app reads the newest `Complete` run and nothing else. A crash halfway therefore
    leaves a partial run that no reader will ever look at, and the previous city stays up.
    """
    check_parameters()
    if manifest.get("version") != DAX_MANIFEST_VERSION:
        raise IngestError(
            f"dax-queries.generated.json is version {manifest.get('version')}, but this notebook "
            f"expects {DAX_MANIFEST_VERSION}. Re-export both from the same commit."
        )

    started = _now()
    window_end = started
    window_start = window_end - _dt.timedelta(days=INGEST_WINDOW_DAYS)
    run_id = str(uuid.uuid4())

    print(f"Probing the semantic model ({METRICS_DATASET_ID}) ...")
    probe = evaluate_dax(manifest["generations"][0]["queries"]["schemaProbe"])
    try:
        generation = pick_generation(manifest, probe)
    except IngestError as error:
        # Export names only, not raw probe fields or telemetry, and still stop before any SQL write.
        tables = probe_tables(probe)
        report = {table: sorted(columns) for table, columns in sorted(tables.items())}
        report_path = Path("builtin/capacity-metrics-schema.json")
        report_path.parent.mkdir(parents=True, exist_ok=True)
        report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
        raise IngestError(
            f"{error} Full table/column schema saved to {report_path}. "
            "Download capacity-metrics-schema.json from notebook Resources > Built-in "
            "to update the adapter; do not rename model tables or skip the schema check."
        ) from error
    print(f"Matched schema generation {generation['name']}.")

    queries = generation["queries"]
    window = {"Start": _iso(window_start), "End": _iso(window_end)}

    inventory_query = queries.get("capacityInventory", queries["capacitySummary"])
    inventory_rows = evaluate_dax(bind_dax_parameters(inventory_query, window))
    ids = capacity_ids(inventory_rows, generation["capacityIdColumn"])
    if not ids:
        # A run with no capacities would be marked Complete and render as an empty atlas, which is
        # indistinguishable from a tenant that genuinely has none. Far more likely is that the
        # notebook identity cannot see the metrics model, so say so instead of publishing nothing.
        raise IngestError(
            f"The capacity inventory returned {len(inventory_rows)} row(s) but no capacity ids in "
            f"column '{generation['capacityIdColumn']}'. Check that this notebook's identity can "
            "read the Capacity Metrics semantic model."
        )
    print(f"{len(inventory_rows)} inventory rows covering {len(ids)} capacities.")
    summary_rows = [] if "capacityInventory" in queries else inventory_rows
    contexts = {}
    if "capacityInventory" in queries:
        for raw in inventory_rows:
            row = normalize_row(raw)
            capacity_id = row.get(generation["capacityIdColumn"])
            region = row.get("Region")
            if not isinstance(capacity_id, str) or not capacity_id or not isinstance(region, str) or not region:
                raise IngestError("Capacity inventory must include a capacity id and its routing Region.")
            if capacity_id in contexts and contexts[capacity_id] != region:
                raise IngestError(f"Capacity {capacity_id} has ambiguous routing regions.")
            contexts[capacity_id] = region

    pending: list[tuple] = []

    def stage(query_name: str, capacity_id: str, rows) -> None:
        for index, row in enumerate(rows):
            stamp = row_timestamp(row, generation["timestampColumn"]) if query_name == "timepoints" else None
            pending.append(
                (
                    str(uuid.uuid4()),
                    run_id,
                    TENANT_ID,
                    query_name,
                    capacity_id,
                    index,
                    stamp,
                    encode_row(row, query_name, index),
                )
            )

    stage("schemaProbe", "", probe_rows_for_generation(probe, generation))

    for capacity_id in ids:
        scoped = {"CapacityId": capacity_id, **window}
        if "capacityInventory" in queries:
            scoped["RegionName"] = contexts[capacity_id]
            rows = evaluate_dax(bind_dax_parameters(queries["capacitySummary"], scoped))
            if len(rows) != 1 or normalize_row(rows[0]).get(generation["capacityIdColumn"]) != capacity_id:
                raise IngestError(f"Capacity summary did not return exactly capacity {capacity_id}.")
            summary_rows.extend(rows)
            print(f"  {capacity_id} capacitySummary: {len(rows)} row ({scoped['RegionName']})")
        for query_name in ("cityItems", "operationFamilies", "timepoints"):
            if query_name in generation.get("unavailableQueries", []):
                print(f"  {capacity_id} {query_name}: unavailable in this schema; no samples inferred")
                continue
            rows = evaluate_dax(bind_dax_parameters(queries[query_name], scoped))
            stage(query_name, capacity_id, rows)
            print(f"  {capacity_id} {query_name}: {len(rows)} rows")

    stage("capacitySummary", "", summary_rows)
    connection = connect_sql()
    run_table = None
    try:
        cursor = connection.cursor()
        run_table = resolve_table(
            cursor,
            "IngestRun",
            [
                "id",
                "tenantId",
                "datasetId",
                "schemaGeneration",
                "status",
                "startedAt",
                "completedAt",
                "windowStart",
                "windowEnd",
                "rowCount",
                "failureMessage",
            ],
        )
        row_table = resolve_table(
            cursor,
            "IngestRow",
            ["id", "runId", "tenantId", "queryName", "capacityId", "rowIndex", "rowTimestamp", "rowJson"],
        )

        cursor.execute(
            f"INSERT INTO {run_table} "
            "([id], [tenantId], [datasetId], [schemaGeneration], [status], [startedAt], [windowStart], "
            "[windowEnd], [rowCount]) VALUES (?, ?, ?, ?, 'Running', ?, ?, ?, 0)",
            run_id,
            TENANT_ID,
            METRICS_DATASET_ID,
            generation["name"],
            started,
            window_start,
            window_end,
        )
        connection.commit()

        cursor.fast_executemany = True
        insert = (
            f"INSERT INTO {row_table} "
            "([id], [runId], [tenantId], [queryName], [capacityId], [rowIndex], [rowTimestamp], [rowJson]) "
            "VALUES (?, ?, ?, ?, ?, ?, ?, ?)"
        )
        for start in range(0, len(pending), BATCH_SIZE):
            cursor.executemany(insert, pending[start : start + BATCH_SIZE])
            connection.commit()

        cursor.execute(
            f"UPDATE {run_table} SET [status] = 'Complete', [completedAt] = ?, [rowCount] = ? WHERE [id] = ?",
            _now(),
            len(pending),
            run_id,
        )
        connection.commit()
        print(f"Run {run_id} complete: {len(pending)} rows.")

        prune_old_runs(cursor, connection, run_table, row_table)
    except Exception as error:
        if run_table is not None:
            _mark_failed(connection, run_table, run_id, error)
        raise
    finally:
        connection.close()

    return run_id


def _mark_failed(connection, run_table: str, run_id: str, error: Exception) -> None:
    """Record why a run stopped, so the app can say so instead of only showing stale data."""
    try:
        cursor = connection.cursor()
        cursor.execute(
            f"UPDATE {run_table} SET [status] = 'Failed', [completedAt] = ?, [failureMessage] = ? WHERE [id] = ?",
            _now(),
            str(error)[:1024],
            run_id,
        )
        connection.commit()
    except Exception as secondary:  # pragma: no cover - best effort only
        print(f"Could not record the failure: {secondary}")


def prune_old_runs(cursor, connection, run_table: str, row_table: str) -> None:
    """Delete all but the newest KEEP_RUNS completed runs, rows first."""
    cursor.execute(
        f"SELECT [id] FROM {run_table} WHERE [tenantId] = ? AND [datasetId] = ? "
        "ORDER BY [startedAt] DESC OFFSET ? ROWS",
        TENANT_ID,
        METRICS_DATASET_ID,
        KEEP_RUNS,
    )
    stale = [row[0] for row in cursor.fetchall()]
    for run_id in stale:
        cursor.execute(f"DELETE FROM {row_table} WHERE [runId] = ?", run_id)
        cursor.execute(f"DELETE FROM {run_table} WHERE [id] = ?", run_id)
        connection.commit()
    if stale:
        print(f"Pruned {len(stale)} older run(s).")

## Run

In [ ]:
run_ingest(MANIFEST)